# Transformer Language Model Architecture

A language model takes as input a batched sequence of integer token IDs (i.e., `torch.Tensor` of shape
(`batch_size`, `sequence_length`)), and returns a (batched) normalized probability distribution over the
vocabulary (i.e., a PyTorch Tensor of shape (`batch_size`, `sequence_length`, `vocab_size`)), where the
predicted distribution is over the next token for each input token. 

When training the language model, we use these next-token predictions to calculate the cross-entropy loss between the actual next token and the predicted next token. When generating text from the language model during inference, we take the predicted next-token distribution from the final time step (i.e., the last item in the sequence) to generate the next token in the sequence (e.g., by taking the token with the highest probability, sampling from the distribution, etc.), add the generated token to the input sequence, and repeat.

In this project, we will build this Transformer language model from scratch.

## Parameter Initialization

Pre-norm transformers are unusually robust to initializations, but they can still have a significant impact on training speed and convergence.

For now, use these approximate initializations (Normal Distribution here refer to the classic Gaussian bell curve distribution):

(a) Linear weights (e.g., in Feed Forward Neural Nets): $N(\mu = 0, \sigma^2 = \frac{2}{d_{in}+d_{out}})$, truncated at $[-3 \sigma, 3 \sigma]$, where $d_{in}$ and $d_{out}$ refer to the input and output dimensions respectively.

(b) Embedding (in the LLM context): $N(\mu = 0, \sigma^2 = 1)$, truncated at $[-3, 3]$.

(c) RMSNorm (in the LLM context): $\mathbb{1}$ -- uniformly 1's.

You should use `torch.nn.init.trunc_normal_` to initialize the truncated normal weights.


## Linear and Embedding Modules, RMS Normalization

### Linear Module

Following most modern LLMs, we will not include a bias term.

#### Coding task for Linear Module:

Implement a `Linear` Python class that inherits from `torch.nn.Module` and performs a linear transformation. Your implementation should follow the following interface. This is intended to resemble the interface of PyTorch’s built-in `nn.Linear` module, except for not having a bias argument or parameter.

- `def __init__(self, in_features, out_features, device=None, dtype=None)` 
  - Construct a linear transformation module. This function should accept the following parameters:
    - `in_features`: `int` 
      final dimension of the input
    - `out_features`: `int` 
      final dimension of the output
    - `device`: `torch.device | None = None` 
      Device to store the parameters on
    - `dtype`: `torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, x: torch.Tensor) -> torch.Tensor` 
  - Apply the linear transformation to the input.

Make sure to:

(i) subclass `nn.Module`

(ii) call the superclass constructor

(iii) construct and store your parameter as W, putting it in an `nn.Parameter`. NOTE: $W \in \mathbb{R}^{d_{out} \times d_{in}}$ is in **row-major** form, where each row corresponds to one output feature. In `forward`, compute the transformation as $x W^T$ (equivalently, `x @ W.T`) over the final input dimension, preserving any leading dimensions of `x`.

(iv) do **not** use `nn.Linear` or `nn.functional.linear`

For initializations, use the settings from above along with `torch.nn.init.trunc_normal_` to initialize the weights.

To test your `Linear` module, implement the test adapter at [adapters.run_linear]. The adapter should load the given weights into your `Linear` module. You can use `Module.load_state_dict` for this purpose. Then, run `uv run pytest -k test_linear` and check that all unit tests pass.

### Embedding Module

Given a sequence of token IDs, the Transformer language model uses an input embedding to convert token IDs to dense vectors, passes the embedded tokens through `num_layers` Transformer blocks, and then applies a learned linear projection (the “output embedding” or “LM head”) to produce the predicted next-token logits.

The first layer of the Transformer is an embedding layer that maps integer token IDs into a vector space of dimension `d_model`. 

We will implement a custom `Embedding` class that inherits from `torch.nn.Module` (so you should not use `nn.Embedding`). The forward method should select the embedding vector for each token ID by indexing into an embedding matrix of shape (`vocab_size`, `d_model`) using a `torch.LongTensor` of token IDs with shape (`batch_size`, `sequence_length`).

#### Coding task for Embedding Module

Implement the `Embedding` class that inherits from `torch.nn.Module` and performs an embedding lookup. Your implementation should use the following interface:

- `def __init__(self, num_embeddings, embedding_dim, device=None, dtype=None)` 
  - Construct an embedding module. This function should accept the following parameters:
    - `num_embeddings`: `int` 
      Size of the vocabulary
    - `embedding_dim` : `int`
      Dimension of the embedding vectors, i.e., `d_model`
    - `device: torch.device | None = None` 
      Device to store the parameters on
    - `dtype: torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, token_ids: torch.Tensor) -> torch.Tensor` 
  - Lookup the embedding vectors for the given token IDs.

Make sure to:

(i) subclass `nn.Module`

(ii) call the superclass constructor

(iii) initialize your embedding matrix as an `nn.Parameter`

(iv) store the embedding matrix with the `d_model` being the final dimension

(v) do **not** use `nn.Embedding` or `nn.functional.embedding`

Again, use the settings from above for initialization, and use `torch.nn.init.trunc_normal_` to initialize the weights. 

To test your implementation, implement the test adapter at [adapters.run_embedding]. Then, run `uv run pytest -k test_embedding` and check that all tests pass.

### Root Mean Square Layer Normalization

We will use root mean square layer normalization.

Given a vector $a \in \mathbb{R}^{d_{model}}$ of activations, `RMSNorm` will scale each activation $a_i$ according to the formula

$\operatorname{RMSNorm}(a_i) = \frac{a_i}{\operatorname{RMS(a)}} \times g_i$,

where $\operatorname{RMS}(a) = \sqrt{ \frac{1}{d_{model}} \times ( \sum_{i=1}^{d_{model}} (a_i)^2 ) + \epsilon }$. 

Here, $g_i$ is a learnable parameter (there are `d_model` such parameters in total), and $\epsilon$ is a hyperparameter often fixed at +1e-05. 

You should upcast your input to `torch.float32` to prevent overflow when you square the input. Overall, your `forward` method should look like:

```python
in_dtype = x.dtype
x = x.to(torch.float32)
# Your code here performing RMSNorm
# ...
result = computed_result
# Return the result in the original dtype
return result.to(in_dtype)
```

#### Coding tasks for RMS Normalization

Implement `RMSNorm` as a `torch.nn.Module`. This Python class should use the following interface:

- `def __init__(self, d_model: int, eps: float = 1e-5, device=None, dtype=None)` 
  - Construct the RMSNorm module. This function should accept the following parameters:
    - `d_model`: `int` 
      Hidden dimension of the model
    - `eps: float = 1e-5` 
      Epsilon value for numerical stability
    - `device: torch.device | None = None` 
      Device to store the parameters on
    - `dtype: torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, x: torch.Tensor) -> torch.Tensor` 
  Process an input tensor of shape (`batch_size`, `sequence_length`, `d_model`) and return a tensor of the same shape.

Note: Remember to upcast your input to `torch.float32` before performing the normalization (and later downcast to the original `dtype`), as described above.
To test your implementation, implement the test adapter at [adapters.run_rmsnorm]. Then, run `uv run pytest -k test_rmsnorm` and make sure all tests pass.

In [ ]:
from __future__ import annotations

import math

import torch


class Linear(torch.nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = torch.nn.Parameter(torch.empty((out_features, in_features), device=device, dtype=dtype))

        std = math.sqrt(2.0 / (in_features + out_features))
        torch.nn.init.trunc_normal_(self.W, mean=0.0, std=std, a=-3.0 * std, b=3.0 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.W.T


class Embedding(torch.nn.Module):
    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.weight = torch.nn.Parameter(torch.empty((num_embeddings, embedding_dim), device=device, dtype=dtype))

        torch.nn.init.trunc_normal_(self.weight, mean=0.0, std=1.0, a=-3.0, b=3.0)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.weight[token_ids]


class RMSNorm(torch.nn.Module):
    def __init__(
        self,
        d_model: int,
        eps: float = 1e-5,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones((d_model,), device=device, dtype=dtype))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        result = x / rms * self.weight.to(torch.float32)
        return result.to(in_dtype)


### How the `Linear`, `Embedding`, and `RMSNorm` classes work

Both classes inherit from `torch.nn.Module`, so PyTorch tracks their parameters through the module machinery. Calling `super().__init__()` sets up that machinery before any parameters are assigned. Each learnable tensor is wrapped in `torch.nn.Parameter`; assigning a `Parameter` to an attribute registers it in the module state dict, includes it in `module.parameters()`, and allows autograd to accumulate gradients into it during backpropagation.

`Linear` stores one parameter, `W`, whose shape is `(out_features, in_features)`. This is a row-major layout for linear-layer weights: each row contains the weights for one output coordinate, and each column corresponds to one input coordinate. If the input tensor `x` has shape `(..., in_features)`, the leading dimensions `...` can be a batch, a sequence, or any other collection of positions. The expression `self.W.T` views the weight matrix with shape `(in_features, out_features)`, and `x @ self.W.T` performs matrix multiplication over only the final input dimension. The result therefore has shape `(..., out_features)`: all leading dimensions are preserved, and the last dimension is replaced by the output-feature dimension. There is no bias tensor, matching the assignment specification.

The `Linear` initializer creates uninitialized storage with `torch.empty`, wraps it as `W`, then fills it in place with `torch.nn.init.trunc_normal_`. The standard deviation is `sqrt(2 / (in_features + out_features))`, so the initialization scale depends on both the fan-in and fan-out of the layer. The lower and upper truncation bounds are `-3 * std` and `3 * std`, which keeps sampled weights within three standard deviations of the zero mean.

`Embedding` stores one parameter, `weight`, whose shape is `(num_embeddings, embedding_dim)`. The first axis is a lookup table over vocabulary IDs: row `i` is the vector assigned to token ID `i`. The final axis is the dense embedding dimension `d_model`, so selecting rows naturally appends that vector dimension to the input token-ID shape. If `token_ids` has shape `(... )`, direct indexing with `self.weight[token_ids]` returns a tensor of shape `(..., embedding_dim)`. For example, a token-ID tensor of shape `(batch_size, sequence_length)` becomes embedded token vectors of shape `(batch_size, sequence_length, embedding_dim)`.

The `Embedding` initializer also uses `torch.empty` followed by `torch.nn.init.trunc_normal_`, but with mean `0`, standard deviation `1`, and fixed bounds `[-3, 3]`. The forward pass does not perform matrix multiplication. It uses tensor indexing to gather rows from the embedding table, so each integer token ID is replaced by the corresponding learned vector while the original token-ID layout is preserved as the leading dimensions of the output tensor.

`RMSNorm` stores one learnable scale parameter, `weight`, whose shape is `(d_model,)`. This corresponds to the vector of $g_i$ values in the RMSNorm formula, one scale value for each coordinate in the final activation dimension. The initializer uses `torch.ones` so the normalization starts as pure root-mean-square rescaling with no learned coordinate-specific change. The module also stores `d_model` and `eps` for clarity and for the normalization denominator.

The `RMSNorm` forward pass first records the input dtype and converts the activations to `torch.float32`. This keeps the square-and-average computation more stable for lower-precision inputs. It then computes `x * x`, averages over the final dimension with `keepdim=True`, adds `eps`, and takes the square root to get a denominator that broadcasts across the original input shape. Dividing `x` by this denominator normalizes each vector over its last dimension, multiplying by `weight` applies the learned per-coordinate scale, and `result.to(in_dtype)` returns the output in the same dtype as the input.
